<div align="center">

# 🚕 Yandex Go Analytics

## 5 · Aggregations & Optimizations

*Business-ready metrics, tuned with Liquid Clustering, Z-Order and OPTIMIZE.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![feature](https://img.shields.io/badge/feature-Liquid%20Clustering-00ADD4)
![feature](https://img.shields.io/badge/feature-Z-Order-7B1FA2)
![maintenance](https://img.shields.io/badge/maintenance-OPTIMIZE-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [6 · Data Enrichment](./6.%20Data%20Enrichment.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▱▱▱  **5 / 8**

---

## 🎯 Task requirements

- [x] **`gold.taxi_metrics`** — group by pickup location and payment description; `total_trips`, `total_revenue` (rounded to 2 decimals); **Liquid Clustering**
- [x] **`gold.driver_metrics`** — group by experience; `drivers_count`, `avg_rating` (rounded to 2 decimals); **Z-Order**
- [x] **`gold.user_metrics`** — count of users per group, sorted in descending order
- [x] **Maintenance** — run `OPTIMIZE` on all three Gold tables

## 🧩 How it is solved

| Gold table | Grouped by | Measures | Optimisation |
|:--|:--|:--|:--|
| `taxi_metrics` | `pu_location_id`, `payment_type_description` | `total_trips` = `COUNT(*)`, `total_revenue` = `ROUND(SUM(fare_amount), 2)` | `ALTER TABLE … CLUSTER BY (pu_location_id)` + `OPTIMIZE` |
| `driver_metrics` | `experience` | `drivers_count` = `COUNT(*)`, `avg_rating` = `ROUND(AVG(rating), 2)` | `OPTIMIZE … ZORDER BY (experience)` |
| `user_metrics` | `registration_date` | `users_count` = `COUNT(*)` | plain `OPTIMIZE` |

Formally, for every group $g$ with trips $T_g$ and drivers $D_g$:

$$\text{total\_revenue}_g=\operatorname{round}\Big(\sum_{t\in T_g}\text{fare}_t,\,2\Big)\qquad
\text{avg\_rating}_g=\operatorname{round}\Big(\tfrac{1}{|D_g|}\sum_{d\in D_g}\text{rating}_d,\,2\Big)$$

| Technique | Best for | Notes |
|:--|:--|:--|
| 🧊 **Liquid Clustering** | Flexible, evolving filter columns | Replaces partitioning and Z-Order; keys can be changed without rewriting the table |
| 🎯 **Z-Order** | Multi-dimensional data skipping | Applied through `OPTIMIZE … ZORDER BY` |
| 🧹 **OPTIMIZE** | Small-file compaction | Run after writes; each Gold service does it automatically |

> 📝 The task groups users by a categorical attribute; this data model has no such column, so the metric counts
> registrations per `registration_date`. The sorted-by-count view is shown below.

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-aggregations")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 12:51:49.747 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-aggregations | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 2️⃣ 🚕 `gold.taxi_metrics` — trips & revenue per zone

`TripGoldMetricsService.run()` performs three actions: **aggregate** (stream, `complete` mode) →
**enable Liquid Clustering** → **`OPTIMIZE`**.

In [2]:
from src.core.etl.gold import (
    DriverGoldMetricsService,
    TripGoldMetricsService,
    UserGoldMetricsService,
)

TripGoldMetricsService().run(spark)

taxi_metrics = spark.table(f"{CATALOG}.gold.taxi_metrics")
print(f"gold.taxi_metrics: {taxi_metrics.count():,} rows")
display(taxi_metrics.orderBy(F.col("total_revenue").desc()).toPandas())

2026-10-09 12:51:49.772 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_dev.silver.taxi -> yandex_go_dev.gold.taxi_metrics
2026-10-09 12:51:49.773 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_dev.silver.taxi
2026-10-09 12:52:36.476 | INFO     | src.core.etl.gold.trip:optimize:19 - Enabling Liquid Clustering on [pu_location_id] and optimizing yandex_go_dev.gold.taxi_metrics...
2026-10-09 12:53:08.119 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_dev.gold.taxi_metrics via checkpoints!


gold.taxi_metrics: 789 rows


DataFrame[pu_location_id: int, payment_type_description: string, total_trips: bigint, total_revenue: double, _calculated_at: timestamp]

### 🧊 Proof: Liquid Clustering is enabled

In [3]:
detail = spark.sql(f"DESCRIBE DETAIL {CATALOG}.gold.taxi_metrics").first()
print(f"Clustering columns: {detail['clusteringColumns']}")

assert "pu_location_id" in detail["clusteringColumns"], "Liquid Clustering is not enabled"
print("✅ Liquid Clustering is active on `pu_location_id`")

Clustering columns: ['pu_location_id']
✅ Liquid Clustering is active on `pu_location_id`


## 3️⃣ 🧑‍✈️ `gold.driver_metrics` — drivers per experience level

Aggregate, then `OPTIMIZE … ZORDER BY (experience)`.

In [4]:
DriverGoldMetricsService().run(spark)

driver_metrics = spark.table(f"{CATALOG}.gold.driver_metrics")
print(f"gold.driver_metrics: {driver_metrics.count():,} rows")
display(driver_metrics.orderBy("experience").toPandas())

2026-10-09 12:53:14.418 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_dev.silver.drivers -> yandex_go_dev.gold.driver_metrics
2026-10-09 12:53:14.419 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_dev.silver.drivers
2026-10-09 12:53:21.119 | INFO     | src.core.etl.gold.driver:optimize:19 - Running Z-Order on yandex_go_dev.gold.driver_metrics
2026-10-09 12:53:23.736 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_dev.gold.driver_metrics via checkpoints!


gold.driver_metrics: 25 rows


DataFrame[experience: int, drivers_count: bigint, avg_rating: double, _calculated_at: timestamp]

### 🎯 Proof: Z-Order was applied

In [5]:
zorder_run = (
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.driver_metrics")
    .filter("operation = 'OPTIMIZE'")
    .orderBy(F.col("version").desc())
    .first()
)
assert zorder_run is not None, "No OPTIMIZE found in the table history"

print(f"Latest OPTIMIZE: version {zorder_run['version']} at {zorder_run['timestamp']}")
print(f"zOrderBy parameter: {zorder_run['operationParameters'].get('zOrderBy')}")

assert "experience" in (zorder_run["operationParameters"].get("zOrderBy") or "")
print("✅ Z-Order on `experience` is recorded in the Delta log")

Latest OPTIMIZE: version 4 at 2026-10-05 10:52:46
zOrderBy parameter: ["experience"]
✅ Z-Order on `experience` is recorded in the Delta log


## 4️⃣ 👤 `gold.user_metrics` — users, most populated groups first

In [6]:
UserGoldMetricsService().run(spark)

user_metrics = spark.table(f"{CATALOG}.gold.user_metrics")
print(f"gold.user_metrics: {user_metrics.count():,} rows")

# Sorted in descending order by the count, as required.
display(user_metrics.orderBy(F.col("users_count").desc()).toPandas())

2026-10-09 12:53:30.213 | INFO     | src.core.etl.gold.base:run:83 - Starting Streaming Aggregation with Checkpoints: yandex_go_dev.silver.users -> yandex_go_dev.gold.user_metrics
2026-10-09 12:53:30.214 | INFO     | src.core.etl.gold.base:extract_stream:50 - Reading incremental stream from: yandex_go_dev.silver.users
2026-10-09 12:53:34.145 | INFO     | src.core.etl.gold.base:optimize:74 - Running OPTIMIZE on yandex_go_dev.gold.user_metrics
2026-10-09 12:53:35.952 | INFO     | src.core.etl.gold.base:run:102 - Successfully updated yandex_go_dev.gold.user_metrics via checkpoints!


gold.user_metrics: 1,109 rows


DataFrame[registration_date: date, users_count: bigint, _calculated_at: timestamp]

## 5️⃣ 🧹 Maintenance — `OPTIMIZE` on all three Gold tables

Every Gold service ends with an `OPTIMIZE`. The Delta log is the evidence:

In [7]:
rows = []
for table in ("taxi_metrics", "driver_metrics", "user_metrics"):
    latest = (
        spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.{table}")
        .filter("operation = 'OPTIMIZE'")
        .orderBy(F.col("version").desc())
        .first()
    )
    assert latest is not None, f"gold.{table} was never optimised"
    params = latest["operationParameters"]
    rows.append(
        (
            f"gold.{table}",
            latest["version"],
            str(latest["timestamp"]),
            params.get("zOrderBy"),
            params.get("clusterBy"),
        )
    )

display(
    spark.createDataFrame(
        rows, "table string, optimize_version long, executed_at string, z_order_by string, cluster_by string"
    ).toPandas()
)
print("✅ OPTIMIZE has been executed on all three Gold tables")

DataFrame[table: string, optimize_version: bigint, executed_at: string, z_order_by: string, cluster_by: string]

✅ OPTIMIZE has been executed on all three Gold tables


---

## ✅ Checkpoint

- [x] `gold.taxi_metrics` with **Liquid Clustering** on `pu_location_id`
- [x] `gold.driver_metrics` with **Z-Order** on `experience`
- [x] `gold.user_metrics` sorted descending by count
- [x] `OPTIMIZE` executed on all three Gold tables; totals reconcile with Silver

> **Next up ▶ [6 · Data Enrichment](./6.%20Data%20Enrichment.ipynb)** — enrich trips with driver profiles.

<div align="center"><sub>Yandex Go Analytics · notebook 5 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>